# Silver Layer: 공간 데이터 통합

vworld 토양 레이어와 OSM 도로 네트워크를 공간 조인합니다.

**실행 환경**: Databricks (Spark + DBFS + Apache Sedona)

In [0]:
# sedona 설치 후 커널 재시작 필수
%pip install apache-sedona==1.7.1

In [0]:
%restart_python

In [0]:
%pip install geopandas

## OSM 도로 네트워크 + vworld 토양 → `silver.walk_features`

vworld 레이어는 행 수가 적어 broadcast 조인으로 셔플 오버헤드를 제거했습니다. 
또한, OSM edge의 p50 length 기준으로 인접 폴리곤과의 거리 임계값을 0.00005도(≈5m)로 설정했습니다.  
경사도 값이 없는 도로는 highway 유형별 기본값으로 보완하고, steps는 수치 경사도 대신 '계단'으로 분류합니다.

In [0]:
from pyspark.sql.functions import expr, first, avg, broadcast, col, when, coalesce, lower, trim, lit, round
from sedona.spark import SedonaContext

spark.conf.set("spark.sql.shuffle.partitions", "200")
sedona = SedonaContext.create(spark)

# 반려견 산책 가능 등급만 필터링
SAFE_HIGHWAYS = [
    "footway", "path", "pedestrian", "residential",
    "living_street", "track", "steps"]

# WKT → 공간 객체 변환
edges_clean = spark.table("bronze.osm_edges") \
    .filter(col("highway").isin(SAFE_HIGHWAYS)) \
    .withColumn("geom_e", expr("ST_GeomFromWKT(geometry)")) \
    .select("id", "length", col("u").alias("start_node"), col("v").alias("end_node"), 
            "surface", "smoothness", "highway", "geometry", "geom_e")

# 공간 조인 전 캐시 고정
edges_clean.cache()
edges_clean.count()
print("OSM 도로 로드 완료")


OSM 도로 로드 완료


In [0]:
# Step 1: 폴리곤 크기 분포
soil_size = spark.table("bronze.vworld_soil_type") \
    .withColumn("geom", expr("ST_GeomFromWKT(geometry)")) \
    .withColumn("area_deg2", expr("ST_Area(geom)")) \
    .withColumn("approx_radius_m", expr("SQRT(area_deg2) * 111000"))

q25, q50, q75 = soil_size.approxQuantile("approx_radius_m", [0.25, 0.5, 0.75], 0.01)
print(f"토양 폴리곤 반경 분포: Q25={q25:.1f}m, Q50={q50:.1f}m, Q75={q75:.1f}m")

# Step 2: ST_Intersects 매칭률 확인
soil_geom = spark.table("bronze.vworld_soil_type") \
    .withColumn("geom_s", expr("ST_GeomFromWKT(geometry)")) \
    .select("DEEPSOIL", "geom_s")

total_edges = edges_clean.count()

matched_count = edges_clean.join(
    broadcast(soil_geom),
    expr("ST_Intersects(geom_e, geom_s)"),
    "inner"
).select("id").distinct().count()

print(f"전체 edge: {total_edges}")
print(f"매칭된 edge: {matched_count}")
print(f"ST_Intersects 매칭률: {matched_count / total_edges * 100:.1f}%")

토양 폴리곤 반경 분포: Q25=124.1m, Q50=206.5m, Q75=406.4m
전체 edge: 143295
매칭된 edge: 10901
ST_Intersects 매칭률: 7.6%


In [0]:
# vworld 토양 데이터 171개 폴리곤으로 해당 지역 일부만 커버
# ST_Intersects 매칭률 7.6% → 데이터 커버리지 한계로 인한 null
# null은 모델에서 결측 처리
soil_type_clean  = spark.table("bronze.vworld_soil_type").select("DEEPSOIL",   expr("ST_GeomFromWKT(geometry)").alias("geom_s"))
slope_clean      = spark.table("bronze.vworld_slope").select("SOILSLOPE",        expr("ST_GeomFromWKT(geometry)").alias("geom_sl"))
gravel_clean     = spark.table("bronze.vworld_gravel").select("SUR_STON",         expr("ST_GeomFromWKT(geometry)").alias("geom_g"))
soil_depth_clean = spark.table("bronze.vworld_soil_depth").select("VLDSOILDEP",   expr("ST_GeomFromWKT(geometry)").alias("geom_sd"))
drainage_clean   = spark.table("bronze.vworld_drainage").select("SOILDRA",        expr("ST_GeomFromWKT(geometry)").alias("geom_d"))

edges_joined = edges_clean \
    .join(broadcast(soil_type_clean),  expr("ST_Intersects(geom_e, geom_s)"),  "left") \
    .join(broadcast(slope_clean),      expr("ST_Intersects(geom_e, geom_sl)"), "left") \
    .join(broadcast(gravel_clean),     expr("ST_Intersects(geom_e, geom_g)"),  "left") \
    .join(broadcast(soil_depth_clean), expr("ST_Intersects(geom_e, geom_sd)"), "left") \
    .join(broadcast(drainage_clean),   expr("ST_Intersects(geom_e, geom_d)"),  "left")

print("공간 조인 완료")

공간 조인 완료


In [0]:
# 집계 단계 파티션 축소 (200 → 64)
spark.conf.set("spark.sql.shuffle.partitions", "64")

# 경사도 범주 문자열 → 구간 중앙값(%)으로 변환
slope_numeric = (
    when(trim(col("SOILSLOPE")) == "0-2%",    1.0)
    .when(trim(col("SOILSLOPE")) == "2-7%",   4.5)
    .when(trim(col("SOILSLOPE")) == "7-15%",  11.0)
    .when(trim(col("SOILSLOPE")) == "15-30%", 22.5)
    .when(trim(col("SOILSLOPE")) == "30-60%", 45.0)
    .when(trim(col("SOILSLOPE")) == "60-100%",80.0)
    .otherwise(None)
)

# geometry를 groupBy 키에서 제외 — 셔플 데이터 볼륨 최소화
silver_df = edges_joined.withColumn("slope_val", slope_numeric).groupBy(
    "id", "start_node", "end_node"
).agg(
    first("geom_e").alias("geom"),
    first("length").alias("length"),
    first("surface").alias("surface"),
    first("smoothness").alias("smoothness"),
    first("highway").alias("highway"),
    first("DEEPSOIL",   True).alias("soil_type"),
    first("SUR_STON",   True).alias("gravel_content"),
    first("VLDSOILDEP", True).alias("soil_depth"),
    first("SOILDRA",    True).alias("drainage_class"),
    avg("slope_val").alias("avg_slope_raw"), 
)

silver_df = silver_df.withColumn("soil_type", coalesce(col("soil_type"), lit("Unknown")))
silver_df.cache()
print("집계 완료")


집계 완료


In [0]:
# 미매칭 구간 결측 보완 — highway 유형별 기본값 부여
silver_df = silver_df.withColumn(
    "avg_slope",
    when(col("highway") == "steps", lit(None))  # steps는 수치 경사도 의미 없음
    .when(col("avg_slope_raw").isNotNull(), col("avg_slope_raw"))
    .otherwise(
        when(col("highway") == "track",                        7.0)
        .when(col("highway").isin("living_street", "path"),    3.5)
        .when(col("highway") == "footway",                     2.5)
        .otherwise(1.5)
    )
).withColumn(
    # 평지(≤3%) / 완만(≤8%) / 급경사(>8%) / 계단
    "slope_type",
    when(col("highway") == "steps",         lit("계단"))
    .when(col("avg_slope").isNull(),         lit("정보없음"))
    .when(col("avg_slope") <= 3.0,           lit("평지"))
    .when(col("avg_slope") <= 8.0,           lit("완만"))
    .otherwise(                              lit("급경사"))
)

silver_df = silver_df \
    .withColumn("avg_slope", round(col("avg_slope"), 1)) \
    .withColumn("surface",   lower(trim(col("surface")))) \
    .withColumn("soil_type", trim(col("soil_type"))) \
    .drop("avg_slope_raw")

silver_df.cache()
print(f"경사도 정제 완료 (행 수: {silver_df.count():,})")


경사도 정제 완료 (행 수: 143,295)


In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

# 22만 건 기준 파일 수 최소화 — Gold에서 읽을 때 스캔 비용 절감
silver_df.withColumn("geometry", expr("ST_AsText(geom)")) \
    .drop("geom") \
    .repartition(10) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.walk_features")

print("✅ silver.walk_features 저장 완료")


✅ silver.walk_features 저장 완료


## 공원 POI → `silver.park`

OSM leisure 데이터에서 공공 접근 가능한 산책 공간만 대상으로 하기 위해 아파트 단지 내 공원은 제거합니다.

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

# OSM에서 아파트 단지가 park으로 오분류된 케이스 제거
EXCLUDE_KEYWORDS = ["아파트", "잠실주공5단지", "아크로힐스", "로렌하우스", "대치에스케이뷰", "래미안대치"]
pattern = "|".join(EXCLUDE_KEYWORDS)

poi_silver = spark.table("bronze.osm_leisure") \
    .withColumn("id", monotonically_increasing_id()) \
    .filter(
        col("leisure").isin("park", "dog_park") &
        (~coalesce(col("name"), lit("")).rlike(pattern))
    ).select(
        "id",
        when(col("name").isNull(), lit("장소명 없음")).otherwise(col("name")).alias("name"),
        "leisure",
        "geometry"
    ).withColumn("geom_p", expr("ST_GeomFromWKT(geometry)"))

poi_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.park")

print("✅ silver.park 저장 완료")
poi_silver.printSchema()


✅ silver.park 저장 완료
root
 |-- id: long (nullable = false)
 |-- name: string (nullable = true)
 |-- leisure: string (nullable = true)
 |-- geometry: string (nullable = true)
 |-- geom_p: geometry (nullable = true)

